#### 01. Find the total matches played, total wins & total losses by each team

In [0]:
%sql
---- Creating new catalog, schema -----
create catalog if not exists sql_youtube_practise;
use catalog sql_youtube_practise;
create schema if not exists sql;
use sql;
show current schema;

catalog,namespace
sql_youtube_practise,sql


In [0]:
%sql
DROP TABLE IF EXISTS icc_world_cup;
create table icc_world_cup
(
Team_1 Varchar(20),
Team_2 Varchar(20),
Winner Varchar(20)
);
INSERT INTO icc_world_cup values('India','SL','India');
INSERT INTO icc_world_cup values('SL','Aus','Aus');
INSERT INTO icc_world_cup values('SA','Eng','Eng');
INSERT INTO icc_world_cup values('Eng','NZ','NZ');
INSERT INTO icc_world_cup values('Aus','India','India');
select * from icc_world_cup;

Team_1,Team_2,Winner
Aus,India,India
India,SL,India
SA,Eng,Eng
SL,Aus,Aus
Eng,NZ,NZ


In [0]:
%sql
with cte as ( 
    select
        Team_1 as team_name,
        case
            when Winner = Team_1 then 1
            else 0
        end as Wins
    from icc_world_cup
    union all
    select
        Team_2 as team_name,
        case
            when Winner = Team_2 then 1
            else 0
        end as Wins
    from icc_world_cup
)
select
    team_name,
    count(1) as total_matches_played,
    sum(Wins) as Wins,
    count(1) - sum(Wins) as Losses
from cte
group by team_name
order by Wins desc;

team_name,total_matches_played,Wins,Losses
India,2,2,0
Aus,2,1,1
Eng,2,1,1
NZ,1,1,0
SA,1,0,1
SL,2,0,2


#### 02. Find the number of first-time customers and repeat customers for each order date.

In [0]:
%sql
DROP TABLE IF EXISTS customer_orders;
CREATE TABLE customer_orders (
    order_id INT,
    customer_id INT,
    order_date DATE,
    order_amount INT
);
INSERT INTO customer_orders VALUES
(1,100,CAST('2022-01-01' AS DATE),2000),
(2,200,CAST('2022-01-01' AS DATE),2500),
(3,300,CAST('2022-01-01' AS DATE),2100),
(4,100,CAST('2022-01-02' AS DATE),2000),
(5,400,CAST('2022-01-02' AS DATE),2200),
(6,500,CAST('2022-01-02' AS DATE),2700),
(7,100,CAST('2022-01-03' AS DATE),3000),
(8,400,CAST('2022-01-03' AS DATE),1000),
(9,600,CAST('2022-01-03' AS DATE),3000);
SELECT * FROM customer_orders;

order_id,customer_id,order_date,order_amount
1,100,2022-01-01,2000
2,200,2022-01-01,2500
3,300,2022-01-01,2100
4,100,2022-01-02,2000
5,400,2022-01-02,2200
6,500,2022-01-02,2700
7,100,2022-01-03,3000
8,400,2022-01-03,1000
9,600,2022-01-03,3000


In [0]:
%sql
with cte as(
    select 
        customer_id,
        min(order_date) as first_visit_date
    from customer_orders
    group by customer_id
),
cte1 as(
    select
        co.*,
        cte.first_visit_date,
        case
            when co.order_date = cte.first_visit_date then 1
            else 0
        end as first_visit_flag,
        case
            when co.order_date != cte.first_visit_date then 1
            else 0
        end as repeat_visit_flag
    from customer_orders as co
    join cte 
    on cte.customer_id = co.customer_id
)
select
    cte1.order_date,
    sum(first_visit_flag) as first_visits,
    sum(repeat_visit_flag) as repeat_visits
from cte1
group by cte1.order_date
order by cte1.order_date;

order_date,first_visits,repeat_visits
2022-01-01,3,0
2022-01-02,2,1
2022-01-03,1,2


###### Using Windows Function

In [0]:
%sql
WITH cte AS (
    SELECT
        *,
        MIN(order_date) OVER (PARTITION BY customer_id) AS first_visit_date
    FROM customer_orders
)
SELECT
    order_date,
    SUM(CASE WHEN order_date = first_visit_date THEN 1 ELSE 0 END) AS first_visits,
    SUM(CASE WHEN order_date != first_visit_date THEN 1 ELSE 0 END) AS repeat_visits
FROM cte
GROUP BY order_date
ORDER BY order_date;

order_date,first_visits,repeat_visits
2022-01-01,3,0
2022-01-02,2,1
2022-01-03,1,2
